In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 56.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=4c79677166190848aeee26677c4119a314338a392f741e3ff8b1f76509c76149
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [4]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.


def get_quantum_random_bits(n):
    backend = BasicSimulator()
    random_bits = []

    # Create the circuit once
    qc = QuantumCircuit(1, 1)
    qc.h(0)           # Superposition state
    qc.measure(0, 0)  # Measurement

    # Transpile the circuit for the backend
    tqc = transpile(qc, backend)

    for _ in range(n):
        # Execute 1 shot to get one random bit
        job = backend.run(tqc, shots=1)
        result = job.result()
        counts = result.get_counts()

        # counts looks like {'0': 1} or {'1': 1}
        # We extract the key '0' or '1' and convert to int
        bit = int(list(counts.keys())[0])
        random_bits.append(bit)

    return random_bits

def alice_prepare_qubits(n):
    # 1. Secretly generate bits and bases
    alice_bits = get_quantum_random_bits(n)
    alice_bases = get_quantum_random_bits(n) # 0 = Rectilinear (Z), 1 = Diagonal (X)

    alice_circuits = []

    # 2. Encode the bits into qubits based on the chosen basis
    for i in range(n):
        # Fresh circuit for each qubit
        qc = QuantumCircuit(1, 1)

        # If bit is 1, apply X gate to change |0> to |1>
        if alice_bits[i] == 1:
            qc.x(0)

        # If basis is 1 (Diagonal), apply H gate to move to the X-basis
        # This turns |0> -> |+> and |1> -> |->
        if alice_bases[i] == 1:
            qc.h(0)
        alice_circuits.append(qc)

    return alice_circuits, alice_bits, alice_bases


def bob_prepare_qubits(n, alice_circuits):
    backend = BasicSimulator()
    bob_bases = get_quantum_random_bits(n) # 0 = Rectilinear (Z), 1 = Diagonal (X)

    bob_bits = []

    for i in range(n):
        qc = alice_circuits[i].copy()

        if bob_bases[i] == 1:
            qc.h(0)

        qc.measure(0, 0)

        # Bob runs the circuit to see if he got a 0 or 1
        tqc = transpile(qc, backend)
        job = backend.run(tqc, shots=1)
        result = job.result()
        counts = result.get_counts()

        # Get the bit result (0 or 1)
        measured_bit = int(list(counts.keys())[0])
        bob_bits.append(measured_bit)

    return bob_bits, bob_bases


def bases_compare(n, alice_bits, bob_bits, alice_bases, bob_bases ):
  # 1. Alice and Bob compare 'alice_bases' and 'bob_bases'
  # 2. Create a 'final_alice_key' and 'final_bob_key'
  # 3. For every index 'i' where alice_bases[i] == bob_bases[i]:
  #    - Keep the bits at that index in their respective final keys
  # 4. Discard all other bits where bases didn't match.

  # final_bob_key = bob_bits
  # final_alice_key = alice_bits
  final_bob_key = []
  final_alice_key = []

  for i in range(n):
    if alice_bases[i] == bob_bases[i]:
      # final_bob_key[i] = None
      # final_alice_key[i] = None

      #safer
      final_alice_key.append(alice_bits[i])
      final_bob_key.append(bob_bits[i])

  return final_alice_key, final_bob_key



def eve_intercept_qubits(n, alice_circuits):
    backend = BasicSimulator()

    # 1. Eve generates her own random bases to guess Alice's
    eve_bases = get_quantum_random_bits(n)
    eve_measured_bits = []
    tampered_channel = []

    for i in range(n):
        # Eve steals a copy of Alice's circuit to measure it
        qc_eve = alice_circuits[i].copy()

        if eve_bases[i] == 1: # X-basis
            qc_eve.h(0)

        qc_eve.measure(0, 0)

        # Execute to get the bit Eve stole
        tqc = transpile(qc_eve, backend)
        result = backend.run(tqc, shots=1).result()
        measured_bit = int(list(result.get_counts().keys())[0])
        eve_measured_bits.append(measured_bit)

        # Eve must send a NEW qubit to Bob based on what she just measured.
        # This is because Alice's original qubit is now "collapsed."
        qc_resend = QuantumCircuit(1, 1)

        # If Eve measured a 1, she applies X
        if measured_bit == 1:
            qc_resend.x(0)

        # If Eve measured in X-basis, she must put her new qubit back in X-basis
        if eve_bases[i] == 1:
            qc_resend.h(0)

        tampered_channel.append(qc_resend)

    return tampered_channel, eve_measured_bits, eve_bases


def check_for_attacker(alice_key, bob_key, threshold=0.15):
    if len(alice_key) == 0:
        return 0, False

    errors = 0
    for a, b in zip(alice_key, bob_key):
        if a != b:
            errors += 1

    bit_error_rate = errors / len(alice_key)

    is_attacked = bit_error_rate > threshold
    return bit_error_rate, is_attacked


In [9]:
# Number of qubits to send
# N_QUBITS = 10

# print("--- Alice's Private Data ---")
# print(f"Alice's Bits:  {a_bits}")
# print(f"Alice's Bases: {a_bases} (0=Z, 1=X)")
# print("\nQuantum Channel contains the prepared circuits.")

# # Visualizing one of the circuits (e.g., the first one)
# print(f"\nCircuit for qubit index 0 (Bit:{a_bits[0]}, Basis:{a_bases[0]}):")
# print(quantum_channel[0].draw())

# b_bits, b_bases = bob_prepare_qubits(N_QUBITS, quantum_channel)

# print("--- Bob's Private Data ---")
# print(f"Bob's Bits:  {b_bits}")
# print(f"Bob's Bases: {b_bases} (0=Z, 1=X)")

# final_alice_key, final_bob_key = bases_compare(N_QUBITS, a_bits, b_bits, a_bases, b_bases)

# print("--- Sifting ---")
# print(f"Alice Final Key:  {final_alice_key}")
# print(f"Bob Final Key: {final_bob_key}")



N = 10  # Number of qubits
print(f"=== Simulation without attacker (N={N}) ===")

# 1. Alice Prepares
q_channel, a_bits, a_bases = alice_prepare_qubits(N)
print(f"Alice's Bits:  {a_bits}")
print(f"Alice's Bases: {a_bases}")

# 2. Bob Measures (Directly from Alice)
b_bits, b_bases = bob_prepare_qubits(N, q_channel)
print(f"Bob's Bits:    {b_bits}")
print(f"Bob's Bases:   {b_bases}")

# 3. Sifting
final_a, final_b = bases_compare(N, a_bits, b_bits, a_bases, b_bases)
print(f"Sifting")
print(f"Alice's Final Key: {final_a}")
print(f"Bob's Final Key:   {final_b}")

# 4. Verification
ber, is_attacked = check_for_attacker(final_a, final_b)
print(f"Bit Error Rate: {ber*100:.1f}%")
print(f"Status: {'⚠️ ATTACK DETECTED' if is_attacked else '✅ SECURE'}")
print("-" * 40)


=== SIMULATION 1: NO ATTACKER (N=10) ===
Alice's Bits:  [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
Alice's Bases: [0, 0, 1, 0, 1, 1, 1, 0, 0, 0]
Bob's Bits:    [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
Bob's Bases:   [0, 0, 1, 0, 0, 1, 0, 1, 0, 1]

[SIFTING COMPLETE]
Alice's Final Key: [1, 0, 0, 1, 1, 1]
Bob's Final Key:   [1, 0, 0, 1, 1, 1]
Bit Error Rate: 0.0%
Status: ✅ SECURE
----------------------------------------


In [16]:
print(f"\n=== Simulation with attacker (Eve) (N={N}) ===")

# 1. Alice Prepares
q_channel, a_bits, a_bases = alice_prepare_qubits(N)
print(f"Alice's Bits:      {a_bits}")
print(f"Alice's Bases:     {a_bases}")

# 2. Eve Intercepts (The Intercept-Resend Attack)
tampered_channel, e_bits, e_bases = eve_intercept_qubits(N, q_channel)
print(f"Eve's Stolen Bits: {e_bits}")
print(f"Eve's Guess Bases: {e_bases}")

# 3. Bob Measures (From the Tampered Channel)
b_bits, b_bases = bob_prepare_qubits(N, tampered_channel)
print(f"Bob's Bits:        {b_bits}")
print(f"Bob's Bases:       {b_bases}")

# 4. Sifting
final_a, final_b = bases_compare(N, a_bits, b_bits, a_bases, b_bases)
print(f"\nSifting")
print(f"Alice's Final Key: {final_a}")
print(f"Bob's Final Key:   {final_b}")

# 5. Detection Logic
# We use a threshold of 0.15 (15%)
ber, is_attacked = check_for_attacker(final_a, final_b, threshold=0.15)
print(f"Bit Error Rate: {ber*100:.1f}%")

if is_attacked:
    print("🚨 ALERT: High Error Rate detected! Eve was listening. Key discarded.")
else:
    print("✅ Key established (No significant interference detected).")


=== SIMULATION 2: WITH ATTACKER (EVE) (N=10) ===
Alice's Bits:      [0, 1, 0, 0, 0, 1, 0, 1, 0, 0]
Alice's Bases:     [0, 1, 1, 0, 1, 0, 0, 0, 1, 0]
Eve's Stolen Bits: [1, 1, 0, 0, 0, 1, 1, 1, 0, 0]
Eve's Guess Bases: [1, 1, 1, 0, 1, 0, 1, 0, 1, 0]
Bob's Bits:        [0, 1, 1, 1, 1, 0, 0, 1, 0, 0]
Bob's Bases:       [0, 0, 0, 1, 0, 1, 0, 0, 1, 0]

[SIFTING COMPLETE]
Alice's Final Key: [0, 0, 1, 0, 0]
Bob's Final Key:   [0, 0, 1, 0, 0]
Bit Error Rate: 0.0%
✅ Key established (No significant interference detected).
